# Customer Churn Prediction: Detailed Implementation Guide

## 1. Problem Statement
**Goal**: Predict whether a Telco customer will churn (leave) or stay based on their demographic and service data.

**Steps to Solve**:
1.  **Data Preparation**: Load data, clean missing values, encode text to numbers, split into Train/Test.
2.  **Model Training**: Train 4 algorithms (k-NN, SVM, Decision Tree, Random Forest).
3.  **Evaluation**: Compare Accuracy, Precision, Recall, F1-Score, and Latency.

**Expected Output**:
*   A clean dataset ready for ML.
*   Trained models with performance metrics.
*   A final recommendation for production usage (<50ms latency).


## 1. Imports and Setup
### Code Explanation
*   **2.1 Definition**: Importing necessary Python libraries.
*   **2.2 Why it is used**: To leverage pre-built functions for data manipulation (`pandas`), math (`numpy`), and Machine Learning (`sklearn`).
*   **2.3 When to use**: At the very beginning of the script.
*   **2.4 Where to use**: Global scope.
*   **2.5 How to use**: `import [library] as [alias]`.
*   **2.6 How it works**: Python loads the module code into memory so we can call its functions.
*   **2.7 Output**: No visible output, but functions become available.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


## 2. Load Data
### Code Explanation
*   **2.1 Definition**: `pd.read_csv` reads a comma-separated values (CSV) file into a DataFrame.
*   **2.2 Why it is used**: To bring data from disk storage into RAM for processing.
*   **2.3 When to use**: When your dataset is stored in a .csv file.
*   **2.4 Where to use**: In the data ingestion phase.
*   **2.5 How to use**: `df = pd.read_csv('path/to/file.csv')`.
*   **2.6 How it works**: It parses the text file, inferring columns and rows, and creates a 2D table object.
*   **2.7 Output**: A `pandas.DataFrame` object.

### Argument Explanation: `filepath_or_buffer`
*   **3.1 Definition**: The path to the file.
*   **3.2 Why it is used**: To locate the data.
*   **3.3 When to use**: Always required.
*   **3.4 Where to use**: First argument.
*   **3.5 How to use**: Pass a string like `'data.csv'`.


In [ ]:
file_name = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(file_name)
df.head()


**Expected Output**: The first 5 rows of the dataset showing columns like `customerID`, `gender`, `Churn`, etc.


## 3. Data Cleaning: TotalCharges
### Code Explanation
*   **2.1 Definition**: Convert `TotalCharges` column to numeric, forcing errors to properties.
*   **2.2 Why it is used**: The column is currently type `object` (text) because of empty strings. We need numbers for math.
*   **2.3 When to use**: When `df.info()` shows a number column as 'object'.
*   **2.4 Where to use**: Data cleaning phase.
*   **2.5 How to use**: `pd.to_numeric(series, errors='coerce')`.
*   **2.6 How it works**: It scans each value. If it's a number, it keeps it. If it's a blank string `" "`, it replaces it with `NaN`.
*   **2.7 Output**: A Series of float values with some NaNs.

### Argument Explanation: `errors='coerce'`
*   **3.1 Definition**: Error handling strategy.
*   **3.2 Why it is used**: To prevent the code from crashing on invalid input.
*   **3.3 When to use**: When you expect some dirty data.
*   **3.4 Where to use**: Inside `to_numeric`.
*   **3.5 How to use**: `errors='coerce'`.


In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()
print('Cleaned TotalCharges and dropped NaNs.')


## 4. Encode Target (Churn)
### Code Explanation
*   **2.1 Definition**: Map 'Yes'/'No' text to 1/0 integers.
*   **2.2 Why it is used**: Machine Learning models require numerical targets.
*   **2.3 When to use**: When your label is categorical.
*   **2.4 Where to use**: Preprocessing.
*   **2.5 How to use**: `df['col'].map({'Yes':1, 'No':0})`.
*   **2.6 How it works**: Replaces every instance of key with value.
*   **2.7 Output**: A Series of 1s and 0s.


In [ ]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['Churn'].value_counts()


## 5. Train-Test Split
### Code Explanation
*   **2.1 Definition**: Splits the dataset into two random subsets.
*   **2.2 Why it is used**: To create a separate 'Test' set to evaluate the model on unseen data.
*   **2.3 When to use**: Before training any model.
*   **2.4 Where to use**: After cleaning and encoding.
*   **2.5 How to use**: `train_test_split(X, y)`.
*   **2.6 How it works**: Randomly shuffles indices and splits data.
*   **2.7 Output**: 4 arrays: `X_train`, `X_test`, `y_train`, `y_test`.

### Argument Explanation: `stratify=y`
*   **3.1 Definition**: Stratified sampling strategy.
*   **3.2 Why it is used**: To preserve class proportions (e.g., if Churn is 10%, Train and Test will both have 10%).
*   **3.3 When to use**: When dealing with classification, especially imbalanced data.
*   **3.4 Where to use**: Argument in `train_test_split`.
*   **3.5 How to use**: `stratify=y`.


In [ ]:
# Drop ID and Churn to get features X
X = df.drop(['customerID', 'Churn'], axis=1)
# Get target y
y = df['Churn']
# One-Hot Encode Categoricals
X = pd.get_dummies(X, drop_first=True)
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')


## 6. Feature Scaling
### Code Explanation
*   **2.1 Definition**: Standardizes features by removing the mean and scaling to unit variance.
*   **2.2 Why it is used**: Algorithms like SVM and k-NN are sensitive to the scale of input data (Age=50 vs Income=50000).
*   **2.3 When to use**: For distance-based algorithms.
*   **2.4 Where to use**: Before training.
*   **2.5 How to use**: `scaler.fit_transform(X_train)`.
*   **2.6 How it works**: Formula: z = (x - mean) / std.
*   **2.7 Output**: A numpy array of scaled values centered at 0.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f'Mean of scaled (should be ~0): {X_train_scaled.mean():.2f}')


## 7. Model Training: Random Forest
### Code Explanation
*   **2.1 Definition**: Training a Random Forest Classifier.
*   **2.2 Why it is used**: It is robust, accurate, and handles non-linear data well.
*   **2.3 When to use**: As a strong baseline for tabular data.
*   **2.4 Where to use**: Modeling phase.
*   **2.5 How to use**: `model.fit(X, y)` and `model.predict(X)`.
*   **2.6 How it works**: Builds 100 decision trees on random subsets of data and averages their votes.
*   **2.7 Output**: A trained model object.

### Argument Explanation: `n_estimators`
*   **3.1 Definition**: Number of trees in the forest.
*   **3.2 Why it is used**: More trees usually mean better stability (but slower).
*   **3.3 When to use**: When defining the model.
*   **3.4 Where to use**: Constructor `RandomForestClassifier`.
*   **3.5 How to use**: `n_estimators=100`.


In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f'Random Forest Accuracy: {acc:.2%}')


**Expected Output**: An accuracy score (e.g., 80.00%) printed to the console.
